In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, concatenate, Masking, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import random, os

# ✅ Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ✅ Configuration
DATA_PATH = "../data/OhioT1DM.csv"
SUBJECT_ID = None    # None -> picks the first subject
LOOKBACK_MIN = 60    # past window (minutes)
PRED_HORIZ_MIN = 30  # forecast horizon (minutes)
STEP_MIN = 5
TEST_FRACTION = 0.2
BATCH_SIZE = 16
EPOCHS = 100
LSTM_UNITS = 64
DROPOUT = 0.2

LOOKBACK_STEPS = LOOKBACK_MIN // STEP_MIN
N_PRED_STEPS = PRED_HORIZ_MIN // STEP_MIN

print("Lookback steps:", LOOKBACK_STEPS,
      "| Prediction steps:", N_PRED_STEPS)

# ✅ Load data
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.sort_values(['id', 'date']).reset_index(drop=True)

# ✅ Select subject
if SUBJECT_ID is None:
    SUBJECT_ID = df['id'].unique()[0]
print("Using subject:", SUBJECT_ID)
sub = df[df['id'] == SUBJECT_ID].copy().reset_index(drop=True)

# ✅ Drop rows missing CGM
sub = sub.dropna(subset=['CGM']).reset_index(drop=True)

# ✅ Auxiliary features
aux_features = [
    'carbs','bolus','basal','galvanic_skin_response','skin_temp','acceleration',
    'workout_intensity','workout_duration','heartrate','air_temp','steps'
]
aux_features = [c for c in aux_features if c in sub.columns]
print("Using AUX:", aux_features)

sub[aux_features] = sub[aux_features].fillna(0.0)
sub['CGM'] = sub['CGM'].astype(float)

# ✅ Build sliding windows
def build_windows(data, lb, ph):
    X_cgm, X_aux, Y, times = [], [], [], []
    for i in range(len(data) - (lb + ph) + 1):
        seq = data.iloc[i:i + lb]
        future = data.iloc[i + lb:i + lb + ph]
        if future['CGM'].isnull().any():
            continue
        X_cgm.append(seq['CGM'].values.reshape(-1,1))
        X_aux.append(seq[aux_features].values)
        Y.append(future['CGM'].values)
        times.append(data.iloc[i + lb]['date'])
    return np.array(X_cgm), np.array(X_aux), np.array(Y), np.array(times)

X_cgm, X_aux, Y, T = build_windows(sub, LOOKBACK_STEPS, N_PRED_STEPS)
print("Windows:", X_cgm.shape)

# ✅ Train/test split
split = int(len(X_cgm)*(1-TEST_FRACTION))
X_cgm_tr, X_cgm_te = X_cgm[:split], X_cgm[split:]
X_aux_tr, X_aux_te = X_aux[:split], X_aux[split:]
Y_tr, Y_te = Y[:split], Y[split:]
T_te = T[split:]

print("Train:", len(X_cgm_tr), "Test:", len(X_cgm_te))

# ✅ Scaling
sc_cgm = StandardScaler().fit(X_cgm_tr.reshape(-1,1))
sc_aux = StandardScaler().fit(X_aux_tr.reshape(-1, X_aux_tr.shape[2]))
sc_y   = StandardScaler().fit(Y_tr.reshape(-1,1))

def scale(Xcgm, Xaux, Yy=None):
    Xcs = sc_cgm.transform(Xcgm.reshape(-1,1)).reshape(Xcgm.shape)
    Xas = sc_aux.transform(Xaux.reshape(-1, Xaux.shape[2])).reshape(Xaux.shape)
    if Yy is not None:
        Ys = sc_y.transform(Yy.reshape(-1,1)).reshape(Yy.shape)
        return Xcs, Xas, Ys
    return Xcs, Xas

Xc_tr_s, Xa_tr_s, Y_tr_s = scale(X_cgm_tr, X_aux_tr, Y_tr)
Xc_te_s, Xa_te_s, Y_te_s = scale(X_cgm_te, X_aux_te, Y_te)

# ✅ Build dual-input LSTM
input_cgm = Input(shape=(LOOKBACK_STEPS,1))
x1 = Masking()(input_cgm)
x1 = LSTM(LSTM_UNITS,return_sequences=True)(x1)
x1 = Dropout(DROPOUT)(x1)
x1 = LSTM(LSTM_UNITS//2)(x1)

input_aux = Input(shape=(LOOKBACK_STEPS,X_aux.shape[2]))
x2 = Masking()(input_aux)
x2 = LSTM(LSTM_UNITS,return_sequences=True)(x2)
x2 = Dropout(DROPOUT)(x2)
x2 = LSTM(LSTM_UNITS//2)(x2)

x = concatenate([x1,x2])
x = Dense(128,activation='relu')(x)
x = Dropout(DROPOUT)(x)
output = Dense(N_PRED_STEPS)(x)

model = Model(inputs=[input_cgm,input_aux], outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
model.summary()

# ✅ Train model
val_split = int(0.9 * len(Xc_tr_s))
history = model.fit(
    [Xc_tr_s[:val_split], Xa_tr_s[:val_split]], Y_tr_s[:val_split],
    validation_data=([Xc_tr_s[val_split:], Xa_tr_s[val_split:]], Y_tr_s[val_split:]),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[EarlyStopping(patience=6,restore_best_weights=True),
               ReduceLROnPlateau(patience=4)],
    verbose=1
)

# ✅ Predict + inverse scale
Y_pred_s = model.predict([Xc_te_s, Xa_te_s], batch_size=BATCH_SIZE)
Y_pred = sc_y.inverse_transform(Y_pred_s.reshape(-1,1)).reshape(Y_pred_s.shape)
Y_true = Y_te

# ✅ Metrics
print("--- METRICS ---")
print("Flattened MSE:", mean_squared_error(Y_true.ravel(), Y_pred.ravel()))
print("Flattened MAE:", mean_absolute_error(Y_true.ravel(), Y_pred.ravel()))

# ✅ Per-step metrics
print("\nStep-wise metrics:")
for i in range(N_PRED_STEPS):
    mse_i = mean_squared_error(Y_true[:,i], Y_pred[:,i])
    mae_i = mean_absolute_error(Y_true[:,i], Y_pred[:,i])
    print(f" +{STEP_MIN*(i+1)} min -> MSE: {mse_i:.2f} | MAE: {mae_i:.2f}")

# ✅ Plot a single sample (mid test)
idx = len(Xc_te_s)//2
anchor_time = T_te[idx]
future_times = [anchor_time + pd.Timedelta(minutes=STEP_MIN*(i+1))
                for i in range(N_PRED_STEPS)]

# Zero-order hold baseline
last_val = X_cgm_te[idx][-1,0]
y_zero = np.repeat(last_val, N_PRED_STEPS)

plt.figure(figsize=(10,4))
plt.plot(future_times, Y_true[idx], 'o-', label='Actual')
plt.plot(future_times, Y_pred[idx], 'o--', label='LSTM')
plt.plot(future_times, y_zero,  'o:', label=f'ZOH baseline ({last_val:.1f})')
plt.grid(); plt.legend()
plt.title(f'Multi-Step CGM Prediction — {SUBJECT_ID} | Anchor: {anchor_time}')
plt.xlabel("Time"); plt.ylabel("CGM (mg/dL)")
plt.gcf().autofmt_xdate()
plt.show()

# ✅ Quick curve for first-step predictions over whole test set
plt.figure(figsize=(12,4))
plt.plot(T_te, Y_true[:,0], '.-', label='Actual +5min')
plt.plot(T_te, Y_pred[:,0], '.--', label='LSTM +5min')
plt.grid(); plt.legend()
plt.title('First-Step Predictions Across Test Timeline')
plt.ylabel("CGM"); plt.xlabel("Time")
plt.gcf().autofmt_xdate()
plt.show()


In [ ]:
# --- Global-model training + per-subject evaluation/plots (paste into a notebook) ---
# - Train a single LSTM model on combined training data from subjects_to_plot subjects
# - Evaluate per subject and plot Actual vs Predicted CGM at t+30min
# - X-axis is minutes (0..30)
# Requirements: put "../data/OhioT1DM.csv" next to your notebook.

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt

# ----------------- parameters -----------------
DATA_PATH = "../data/OhioT1DM.csv"
n_in = 24           # input window in steps (12 * 5min = 60 min history)
n_out = 1           # output horizon in steps (6 * 5min = 30 min ahead)
epochs = 50
batch_size = 16
subjects_to_plot = 3
# ------------------------------------------------

# helper: sliding windows
def make_sequences(values, n_in, n_out):
    X, y = [], []
    for i in range(len(values) - n_in - n_out + 1):
        X.append(values[i:i+n_in])
        # y is the CGM column (first column)
        y.append(values[i+n_in:i+n_in+n_out, 0])
    return np.array(X), np.array(y)

# read data
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
feature_cols = ["CGM","carbs","bolus","basal","galvanic_skin_response","skin_temp",
                "acceleration","workout_intensity","workout_duration","heartrate","air_temp","steps"]

ids = df['id'].unique()[:subjects_to_plot]

# --- first pass: collect raw per-subject arrays so we can fit global scalers ---
raw_per_subject = {}
for sid in ids:
    sub = df[df['id']==sid].sort_values('date').copy()
    sub = sub.set_index('date')
    # reindex to consistent 5-min sampling
    full_idx = pd.date_range(sub.index.min(), sub.index.max(), freq='5T')
    sub = sub.reindex(full_idx)
    # sensible fills
    sub['CGM'] = sub['CGM'].ffill().bfill()
    sub[['carbs','bolus','basal','workout_intensity','workout_duration','steps']] = \
        sub[['carbs','bolus','basal','workout_intensity','workout_duration','steps']].fillna(0)
    sub[['galvanic_skin_response','skin_temp','acceleration','heartrate','air_temp']] = \
        sub[['galvanic_skin_response','skin_temp','acceleration','heartrate','air_temp']].ffill().bfill().fillna(0)

    data = sub[feature_cols].values.astype(float)   # shape (T, n_features)
    raw_per_subject[sid] = {'data': data, 'full_idx': full_idx}

# check we have data
if len(raw_per_subject) == 0:
    raise RuntimeError("No subject data found for the chosen ids.")

# concatenate across subjects to fit global scalers (CGM scaler + "others" scaler)
all_cg = np.vstack([raw_per_subject[s]['data'][:, 0:1] for s in raw_per_subject])
all_others = np.vstack([raw_per_subject[s]['data'][:, 1:] for s in raw_per_subject])

cg_scaler = MinMaxScaler().fit(all_cg)
others_scaler = MinMaxScaler().fit(all_others)

# --- second pass: create sequences, split per-subject (chronological), and collect global training set ---
X_train_list = []
y_train_list = []
per_subject_test = {}   # store X_test, y_test, times_test for each subject

for sid, info in raw_per_subject.items():
    data = info['data']
    full_idx = info['full_idx']

    # apply global scalers
    cg_scaled = cg_scaler.transform(data[:, 0:1])
    others_scaled = others_scaler.transform(data[:, 1:])
    data_scaled = np.hstack([cg_scaled, others_scaled])

    # make sequences
    X, y = make_sequences(data_scaled, n_in, n_out)
    if len(X) == 0:
        print(f"Warning: subject {sid} has insufficient data for windows -> skipping.")
        continue

    # timestamps correspond to the final predicted step (t + (n_out-1)*5min)
    timestamps = full_idx[n_in : n_in + len(y)] + pd.to_timedelta(5*(n_out-1), unit='m')

    # chronological split for each subject: first 70% -> train, last 30% -> test
    split = int(0.7 * len(X))
    if split < 1:
        # too small to get training portion
        print(f"Warning: subject {sid} has too few windows for training/test split -> skipping.")
        continue

    X_train_list.append(X[:split])
    y_train_list.append(y[:split])

    per_subject_test[sid] = {
        'X_test': X[split:],
        'y_test': y[split:],
        'times_test': timestamps[split:]
    }

# build global training arrays
if len(X_train_list) == 0:
    raise RuntimeError("No training data collected across subjects (check subject selection and data).")

X_train_all = np.vstack(X_train_list)
y_train_all = np.vstack(y_train_list)

print(f"Global training samples: {len(X_train_all)}   (n_in={n_in}, n_out={n_out})")

# --- build single global model and train once ---
n_features = X_train_all.shape[2]

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(n_in, n_features)),
    LSTM(64, return_sequences=False),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(n_out)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# shuffle the global training set (we combined chronological train parts from each subject)
history = model.fit(
    X_train_all, y_train_all,
    epochs=epochs,
    batch_size=batch_size,
    verbose=1,
    shuffle=True,
    validation_split=0.1,
    callbacks=callbacks
)

# --- per-subject evaluation and plotting ---
def inv_cgm_with_cg_scaler(scaled_vals):
    # scaled_vals shape: (n_samples, n_out)
    out = []
    for step_idx in range(scaled_vals.shape[1]):
        col = scaled_vals[:, step_idx].reshape(-1, 1)
        inv = cg_scaler.inverse_transform(col)[:, 0]
        out.append(inv)
    return np.stack(out, axis=1)  # shape (n_samples, n_out)

plt.figure(figsize=(10, 4 * len(per_subject_test)))
for idx_i, sid in enumerate(per_subject_test.keys()):
    info = per_subject_test[sid]
    X_test = info['X_test']
    y_test = info['y_test']
    times_test = info['times_test']

    if len(X_test) == 0:
        print(f"No test samples for subject {sid}, skipping plot.")
        continue

    y_pred = model.predict(X_test, batch_size=batch_size)

    # invert CGM scaling cleanly
    y_test_inv = inv_cgm_with_cg_scaler(y_test)
    y_pred_inv = inv_cgm_with_cg_scaler(y_pred)

    # take the final forecast step (t + 30 min)
    actual_30 = y_test_inv[:, -1]
    pred_30 = y_pred_inv[:, -1]

    # minutes since start of this subject's test portion
    t0 = times_test[0]
    minutes = np.array([(t - t0).total_seconds() / 60.0 for t in times_test])

    # only show up to 30 minutes (mask)
    mask = minutes <= 30.0
    if mask.sum() == 0:
        # fallback: plot up to first min(30, len(points)) points (use their minutes)
        idx_plot = np.arange(min(30, len(minutes)))
        minutes_plot = minutes[idx_plot]
        actual_plot = actual_30[idx_plot]
        pred_plot = pred_30[idx_plot]
    else:
        minutes_plot = minutes[mask]
        actual_plot = actual_30[mask]
        pred_plot = pred_30[mask]

    ax = plt.subplot(len(per_subject_test), 1, idx_i+1)
    ax.plot(minutes_plot, actual_plot, label='Actual CGM (t+30min)', linewidth=1)
    ax.plot(minutes_plot, pred_plot, label='Predicted CGM (t+30min)', linewidth=1)
    ax.set_ylabel('BG (mg/dL)')
    ax.set_xlabel('Minutes since start of test segment')
    ax.set_title(f'Subject id={sid}   (history {n_in*5} min -> predict {n_out*5} min ahead)')
    ax.set_xlim(0, 30)   # enforce 0..30 minutes
    ax.legend()
    ax.grid(True)

plt.tight_layout()
# ---------------------------------------------------------
